# Instalamos las dependencias

In [1]:
pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Lectura del PDF

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Segmentar texto e chunks

In [3]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Texto -> Vector

In [4]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Convertirmos chunks en vectores

In [5]:
chunk_vectors = []
chunk_vectors_len = []
for chunk in chunks:
    vector = text_to_vector(chunk)
    chunk_vectors.append(vector)
    chunk_vectors_len.append(len(vector))

print(len(chunk_vectors))
print(chunk_vectors_len)

0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
ERROR:tornado.general:SEND Error: Host unreachable


KeyboardInterrupt: 

# Crear arbol KD Tree con auto balanceo

In [ ]:
!apt-get install swig

# Header KDTree

In [ ]:
%%file kdtree.h
#ifndef KDTREE_H
#define KDTREE_H

#include <vector>

struct Node {
    std::vector<double> point;
    Node* left;
    Node* right;

    Node(std::vector<double> p) : point(p), left(nullptr), right(nullptr) {}
};

class KDTree {
public:
    KDTree();
    ~KDTree();

    void insert(std::vector<double> point);
    bool search(std::vector<double> point);

private:
    Node* root;
    Node* insertRec(Node* node, std::vector<double> point, int depth);
    bool searchRec(Node* node, std::vector<double> point, int depth);
    void freeTree(Node* node);
};

#endif


# Implementación KDTree

In [ ]:
%%file kdtree.cpp
#include "kdtree.h"
#include <iostream>

KDTree::KDTree() : root(nullptr) {}

KDTree::~KDTree() {
    freeTree(root);
}

void KDTree::insert(std::vector<double> point) {
    root = insertRec(root, point, 0);
}

bool KDTree::search(std::vector<double> point) {
    return searchRec(root, point, 0);
}

Node* KDTree::insertRec(Node* node, std::vector<double> point, int depth) {
    if (node == nullptr) {
        return new Node(point);
    }

    int axis = depth % point.size();
    if (point[axis] < node->point[axis]) {
        node->left = insertRec(node->left, point, depth + 1);
    } else {
        node->right = insertRec(node->right, point, depth + 1);
    }

    return node;
}

bool KDTree::searchRec(Node* node, std::vector<double> point, int depth) {
    if (node == nullptr) return false;
    if (node->point == point) return true;

    int axis = depth % point.size();
    if (point[axis] < node->point[axis]) {
        return searchRec(node->left, point, depth + 1);
    } else {
        return searchRec(node->right, point, depth + 1);
    }
}

void KDTree::freeTree(Node* node) {
    if (node) {
        freeTree(node->left);
        freeTree(node->right);
        delete node;
    }
}


# Interfaz de SWIG

In [ ]:
%%file kdtree.i
%module kdtree

%{
#include "kdtree.h"
%}

%include "std_vector.i"
%template(vector_double) std::vector<double>;

%include "kdtree.h"


# Generar Wrapper

In [15]:
!swig -c++ -python kdtree.i

!g++ -shared -fPIC -I/usr/include/python3.10 -o _kdtree.so kdtree_wrap.cxx kdtree.cpp

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
25168.34s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Unable to find file 'kdtree.i'.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
25173.96s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


clang++: error: no such file or directory: 'kdtree_wrap.cxx'
clang++: error: no such file or directory: 'kdtree.cpp'
clang++: error: no input files


# Realizar búsqueda